# Ingestion Pipeline: Docling to Chroma Cloud with server side hybrid search

**What changed vs the previous version**

The old pipeline had a correctness bug. The manifest skipped already ingested PDFs,
so `all_langchain_docs` only ever held chunks from *new or changed* files. Dense search
read from Chroma Cloud and therefore saw the whole corpus, but `BM25Retriever.from_documents()`
was built in memory from that partial list. On any incremental run the lexical half of the
hybrid retriever was searching a tiny fraction of the corpus, and after a runtime restart with
no new PDFs it was searching nothing at all.

The fix is to stop maintaining a client side lexical index. Chroma Cloud supports a sparse
vector index alongside the dense one, populated automatically from the document text on write,
and `Rrf` fuses the two rankings server side. BM25 now lives next to the embeddings, covers
every chunk ever ingested, and survives runtime restarts. `EnsembleRetriever` and `rank_bm25`
are gone entirely.

In [1]:
# rank_bm25 and langchain-classic are no longer needed: BM25 is now a Chroma sparse index.
# snowballstemmer is required by ChromaBm25EmbeddingFunction for tokenization.
!pip install -q docling transformers langchain langchain-core chromadb snowballstemmer pymupdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 814.7/814.7 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Point this at wherever your research papers live in Drive
PDF_FOLDER = "/content/drive/MyDrive/company_docs"

pdf_paths = sorted(Path(PDF_FOLDER).glob("*.pdf"))
print(f"Found {len(pdf_paths)} PDFs:")
for p in pdf_paths:
    print(f"  {p.name}")

Mounted at /content/drive
Found 11 PDFs:
  acceptable-use-policy.pdf
  benefits-overview.pdf
  code-of-conduct.pdf
  expense-reimbursement-policy.pdf
  holiday-schedule.pdf
  information-security-policy.pdf
  onboarding-guide.pdf
  performance-review-policy.pdf
  pto-and-leave-policy.pdf
  public_counsel_employee_handbook.pdf
  remote-work-policy.pdf


In [4]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["CHROMA_API_KEY"] = userdata.get("CHROMA_API_KEY")
os.environ["CHROMA_TENANT"] = userdata.get("CHROMA_TENANT")
os.environ["COHERE_API_KEY"] = userdata.get("COHERE_API_KEY")

## Config

Collection name is bumped deliberately. A sparse vector index is part of the collection
schema and is applied at creation time, so pointing `get_or_create_collection` at the old
`research_papers` collection would hand back the existing schema with no sparse index and
hybrid search would fail. See the migration note at the bottom of the notebook.

In [5]:
CHROMA_DATABASE = "production-rag"

# New collection: the sparse index is a schema level property fixed at creation time,
# so the old dense only collection cannot be upgraded in place.
COLLECTION_NAME = "research_papers_hybrid"

# Metadata key the sparse (BM25) vectors are stored under. Any name works, it just has to
# match between the schema definition and the Knn(key=...) used at query time.
SPARSE_KEY = "sparse_embedding"

EMBEDDING_MODEL = "text-embedding-3-small"

# Manifest lives in Drive, not in the ephemeral Colab filesystem. Previously a runtime
# restart wiped it and every PDF was reprocessed from scratch on the next run.
MANIFEST_PATH = Path(PDF_FOLDER) / ".ingestion_manifest.json"

In [6]:
!pip install PyMuPDF

In [7]:
import fitz  # PyMuPDF
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode


def needs_ocr(pdf_path):
    """Check for a real text layer before deciding whether to run OCR.
    A batch of research papers can mix native digital PDFs and scanned ones,
    so this can't be a blanket on/off decision like it was for the single-paper flow."""
    doc = fitz.open(pdf_path)
    try:
        for page in doc:
            if page.get_text().strip():
                return False
        return True
    finally:
        doc.close()  # fitz keeps the file handle open otherwise, which bites on large batches


def build_converter(ocr_needed):
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = ocr_needed
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options.mode = TableFormerMode.FAST
    pipeline_options.do_formula_enrichment = True  # papers have real math
    pipeline_options.do_picture_classification = False
    pipeline_options.do_picture_description = False

    return DocumentConverter(
        format_options={"pdf": PdfFormatOption(pipeline_options=pipeline_options)}
    )

In [8]:
from docling.chunking import HybridChunker
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

chunker = HybridChunker(
    tokenizer=tokenizer,
    max_tokens=512,
    merge_peers=True,
)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## Manifest

The old notebook had a cell that bootstrapped the manifest with every file hashed
*before* any ingestion ran. On a fresh environment that cell caused the ingestion loop
below to skip all of them, producing an empty corpus. It has been removed. The manifest
is now only ever written by the ingestion loop, after a file actually succeeds.

In [9]:
import hashlib
import json

def load_manifest():
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text())
    return {}

def save_manifest(manifest):
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))

def file_hash(pdf_path):
    # hash the raw bytes, not just the filename, so an edited PDF gets reprocessed
    return hashlib.sha256(pdf_path.read_bytes()).hexdigest()

def clean_metadata(meta):
    """Chroma rejects None metadata values. page_no is legitimately None for chunks
    with no provenance, so drop those keys rather than letting the upsert fail."""
    return {k: v for k, v in meta.items() if v is not None}

print(f"Manifest: {MANIFEST_PATH}")
print(f"Currently tracking {len(load_manifest())} files.")

Manifest: /content/drive/MyDrive/company_docs/.ingestion_manifest.json
Currently tracking 11 files.


In [10]:
from langchain_core.documents import Document
import time

manifest = load_manifest()

all_langchain_docs = []
failed_files = []
skipped_files = []

for pdf_path in pdf_paths:
    current_hash = file_hash(pdf_path)

    if manifest.get(pdf_path.name) == current_hash:
        print(f"Skipping {pdf_path.name} (already ingested, unchanged)")
        skipped_files.append(pdf_path.name)
        continue

    start = time.time()
    print(f"Processing {pdf_path.name}...")

    try:
        ocr_needed = needs_ocr(pdf_path)
        converter = build_converter(ocr_needed)
        result = converter.convert(str(pdf_path))

        chunks = list(chunker.chunk(dl_doc=result.document))

        for chunk in chunks:
            contextualized_text = chunker.contextualize(chunk=chunk)

            chunk_id = hashlib.sha256(
                f"{pdf_path.name}:{contextualized_text}".encode()
            ).hexdigest()

            page_no = (
                chunk.meta.doc_items[0].prov[0].page_no
                if chunk.meta.doc_items and chunk.meta.doc_items[0].prov
                else None
            )

            all_langchain_docs.append(
                Document(
                    page_content=contextualized_text,
                    metadata=clean_metadata({
                        "source": pdf_path.name,
                        "headings": ", ".join(chunk.meta.headings) if chunk.meta.headings else "",
                        "page_no": page_no,
                        "chunk_id": chunk_id,
                        "ocr_used": ocr_needed,
                    }),
                )
            )

        elapsed = time.time() - start
        print(f"  {len(chunks)} chunks, OCR={'on' if ocr_needed else 'off'}, {elapsed:.1f}s")

        # Staged, not committed. The manifest is only written after the upsert cell
        # succeeds, otherwise a failed upsert would leave files marked as ingested
        # that never actually reached Chroma.
        manifest[pdf_path.name] = current_hash

    except Exception as e:
        print(f"  FAILED: {e}")
        failed_files.append((pdf_path.name, str(e)))
        continue

print(f"\nTotal new chunks: {len(all_langchain_docs)}")
print(f"Skipped (unchanged): {len(skipped_files)}")
if failed_files:
    print(f"\n{len(failed_files)} file(s) failed:")
    for name, err in failed_files:
        print(f"  {name}: {err}")

Skipping acceptable-use-policy.pdf (already ingested, unchanged)
Skipping benefits-overview.pdf (already ingested, unchanged)
Skipping code-of-conduct.pdf (already ingested, unchanged)
Skipping expense-reimbursement-policy.pdf (already ingested, unchanged)
Skipping holiday-schedule.pdf (already ingested, unchanged)
Skipping information-security-policy.pdf (already ingested, unchanged)
Skipping onboarding-guide.pdf (already ingested, unchanged)
Skipping performance-review-policy.pdf (already ingested, unchanged)
Skipping pto-and-leave-policy.pdf (already ingested, unchanged)
Skipping public_counsel_employee_handbook.pdf (already ingested, unchanged)
Skipping remote-work-policy.pdf (already ingested, unchanged)

Total new chunks: 0
Skipped (unchanged): 11


## Chroma Cloud collection with dense + sparse indexes

Two indexes are declared on the schema:

- **Dense** via `VectorIndexConfig`, using OpenAI embeddings, sourced from the document text.
- **Sparse** via `SparseVectorIndexConfig`, using `ChromaBm25EmbeddingFunction`, also sourced
  from the document text and stored under the `sparse_embedding` metadata key.

Both are generated by Chroma on write. That is the whole point of the change: you hand Chroma
`documents=[...]` and it maintains the BM25 postings itself, so the lexical index is always in
sync with the dense index and always covers the full corpus regardless of what this particular
run happened to process.

Because the embedding function is attached to the collection, `Knn(query="some text")` embeds
the query for you at search time. There is no client side embedding call and no LangChain
`Chroma` wrapper involved anymore.

In [11]:
import chromadb
from chromadb import Schema, VectorIndexConfig, SparseVectorIndexConfig, K
from chromadb.utils.embedding_functions import (
    OpenAIEmbeddingFunction,
    ChromaBm25EmbeddingFunction,
)

client = chromadb.CloudClient(
    api_key=os.environ["CHROMA_API_KEY"],
    tenant=os.environ["CHROMA_TENANT"],
    database=CHROMA_DATABASE,
)

dense_ef = OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    model_name=EMBEDDING_MODEL,
)

# Runs locally, no API key, no extra latency budget. b/k are the standard BM25 knobs.
# avg_doc_length should roughly match your chunk size in tokens: HybridChunker is capped at
# 512 and contextualized chunks land well below that, so 256 (the default) is a sane starting
# point. Changing it later invalidates the existing sparse vectors, so tune it before you
# backfill, not after.
sparse_ef = ChromaBm25EmbeddingFunction(
    k=1.2,
    b=0.75,
    avg_doc_length=256.0,
    token_max_length=40,
)

schema = (
    Schema()
    .create_index(
        config=VectorIndexConfig(
            space="cosine",
            embedding_function=dense_ef,
        )
    )
    .create_index(
        config=SparseVectorIndexConfig(
            source_key=K.DOCUMENT,
            embedding_function=sparse_ef,
        ),
        key=SPARSE_KEY,
    )
)

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    schema=schema,
)

print(f"Collection '{COLLECTION_NAME}' ready. Current count: {collection.count()}")

Collection 'research_papers_hybrid' ready. Current count: 350


## Upsert

`upsert` with deterministic `chunk_id` values keeps re-runs idempotent, same as before.
The difference is that a single write now populates both indexes, so there is no second
code path that can silently fall out of sync.

Batched because a large first run will otherwise push a very large payload and a very large
embedding request in one shot.

In [12]:
BATCH_SIZE = 100

if all_langchain_docs:
    ids = [d.metadata["chunk_id"] for d in all_langchain_docs]
    documents = [d.page_content for d in all_langchain_docs]
    metadatas = [d.metadata for d in all_langchain_docs]

    for i in range(0, len(ids), BATCH_SIZE):
        collection.upsert(
            ids=ids[i:i + BATCH_SIZE],
            documents=documents[i:i + BATCH_SIZE],
            metadatas=metadatas[i:i + BATCH_SIZE],
        )
        print(f"  upserted {min(i + BATCH_SIZE, len(ids))}/{len(ids)}")

    # Commit the manifest only now that the data is actually in Chroma.
    save_manifest(manifest)
    print(f"\nUpserted {len(ids)} chunks. Collection count: {collection.count()}")
else:
    print("No new or changed PDFs to embed, nothing to upsert.")
    print(f"Collection count: {collection.count()}")

No new or changed PDFs to embed, nothing to upsert.
Collection count: 350


## Hybrid search with RRF

`Rrf` fuses the dense and sparse rankings by rank position rather than raw score, which is
what you want here since cosine distance and BM25 scores are on completely different scales.

Three details that matter:

- `return_rank=True` is mandatory on every `Knn` inside `Rrf`. Without it the expression fuses
  raw distances and the ranking is silently wrong.
- `limit` on each `Knn` is the candidate pool each retriever contributes, not the final result
  count. 100 to 500 is the usual range; the final cut is the outer `.limit()`.
- `default` lets a document that appears in only one of the two rankings still be scored.
  Without it you get an implicit intersection, which defeats the purpose of hybrid retrieval:
  an exact keyword hit that the dense retriever missed would be dropped.

`weights=[0.6, 0.4]` preserves the dense/lexical balance the old `EnsembleRetriever` used.

In [13]:
from chromadb import Search, Knn, Rrf

CANDIDATE_POOL = 200   # per-retriever candidate depth
MISSING_RANK = 1000    # rank assigned to docs absent from one of the two rankings

def hybrid_search(query, k=5, where=None, dense_weight=0.6, sparse_weight=0.4):
    """Server-side hybrid retrieval. Dense semantic + BM25 lexical, fused with RRF."""
    hybrid_rank = Rrf(
        ranks=[
            Knn(
                query=query,
                return_rank=True,
                limit=CANDIDATE_POOL,
                default=MISSING_RANK,
            ),
            Knn(
                query=query,
                key=SPARSE_KEY,
                return_rank=True,
                limit=CANDIDATE_POOL,
                default=MISSING_RANK,
            ),
        ],
        weights=[dense_weight, sparse_weight],
        k=60,
    )

    search = Search().rank(hybrid_rank).limit(k).select(K.DOCUMENT, K.SCORE, K.METADATA)
    if where is not None:
        search = search.where(where)

    return collection.search(search).rows()[0]

## LangChain retriever wrapper

Thin `BaseRetriever` over the search above so anything downstream that expects
`.invoke(query) -> list[Document]` keeps working unchanged. The RRF score is carried
through on metadata. Note that RRF scores are negative and *lower is better*, which is the
opposite of what most LangChain rerankers assume, so do not sort on it naively downstream.

In [14]:
from typing import Any, List, Optional
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from pydantic import ConfigDict


class ChromaHybridRetriever(BaseRetriever):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    collection: Any
    sparse_key: str = "sparse_embedding"
    k: int = 5
    dense_weight: float = 0.6
    sparse_weight: float = 0.4
    candidate_pool: int = 200
    rrf_k: int = 60
    missing_rank: int = 1000
    where: Optional[Any] = None

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        hybrid_rank = Rrf(
            ranks=[
                Knn(query=query, return_rank=True,
                    limit=self.candidate_pool, default=self.missing_rank),
                Knn(query=query, key=self.sparse_key, return_rank=True,
                    limit=self.candidate_pool, default=self.missing_rank),
            ],
            weights=[self.dense_weight, self.sparse_weight],
            k=self.rrf_k,
        )

        search = (
            Search()
            .rank(hybrid_rank)
            .limit(self.k)
            .select(K.DOCUMENT, K.SCORE, K.METADATA)
        )
        if self.where is not None:
            search = search.where(self.where)

        rows = self.collection.search(search).rows()[0]

        return [
            Document(
                page_content=row["document"],
                metadata={**(row.get("metadata") or {}), "rrf_score": row.get("score")},
            )
            for row in rows
        ]


hybrid_retriever = ChromaHybridRetriever(collection=collection, sparse_key=SPARSE_KEY, k=5)

In [15]:
results = hybrid_retriever.invoke("What are Acme Corp Core Values?")
for doc in results:
    m = doc.metadata
    print(f"{m.get('source')} - page {m.get('page_no')} - {m.get('headings')}")
    print(f"   rrf_score={m.get('rrf_score'):.5f}")

code-of-conduct.pdf - page 1 - 2. Core Values
   rrf_score=-0.01667
remote-work-policy.pdf - page 2 - 5.1 Core Hours
   rrf_score=-0.01608
remote-work-policy.pdf - page 7 - Acme Corp - Official Policy Document
   rrf_score=-0.01517
code-of-conduct.pdf - page 1 - Acme Corp - Code of Conduct
   rrf_score=-0.01517
information-security-policy.pdf - page 1 - Acme Corp - Information Security Policy
   rrf_score=-0.01502


## Sanity check

The bug this notebook fixes was invisible: dense results looked fine, so hybrid results looked
fine, while the lexical half quietly contributed nothing. Run the two legs separately after an
incremental ingest and confirm the sparse leg returns hits from papers that were *skipped*
this run. If it comes back empty, the sparse index was not populated and you are back to
dense-only retrieval.

In [16]:
def _leg(query, key=None, k=5):
    rank = Knn(query=query, key=key) if key else Knn(query=query)
    search = Search().rank(rank).limit(k).select(K.SCORE, K.METADATA)
    return collection.search(search).rows()[0]

q = "What are Acme Corp Core Values?"

print(f"Collection count: {collection.count()}")
print(f"Files skipped this run: {len(skipped_files)}")

print("\nDense only:")
for r in _leg(q):
    print(f"  {r['metadata'].get('source')} p{r['metadata'].get('page_no')}  score={r['score']:.4f}")

print("\nSparse (BM25) only:")
sparse_rows = _leg(q, key=SPARSE_KEY)
for r in sparse_rows:
    print(f"  {r['metadata'].get('source')} p{r['metadata'].get('page_no')}  score={r['score']:.4f}")

if not sparse_rows:
    print("  WARNING: sparse leg returned nothing. Check that the collection was created "
          "with the sparse index in its schema.")

Collection count: 350
Files skipped this run: 11

Dense only:
  code-of-conduct.pdf p1  score=0.1140
  code-of-conduct.pdf p1  score=0.3361
  code-of-conduct.pdf p1  score=0.3416
  remote-work-policy.pdf p2  score=0.3830
  code-of-conduct.pdf p2  score=0.4025

Sparse (BM25) only:
  code-of-conduct.pdf p1  score=-9.9424
  remote-work-policy.pdf p2  score=-7.3747
  code-of-conduct.pdf p4  score=-6.9189
  remote-work-policy.pdf p7  score=-5.5530
  information-security-policy.pdf p3  score=-5.5293


### Enhanced RAG Generation

This section defines a custom prompt and a retrieval chain that leverages the `hybrid_retriever` we built above. It is designed to use the metadata fields (source, page number, and headings) to provide grounded answers.

In [17]:
!pip install langchain-cohere cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 63.3 MB/s eta 0:00:00


In [19]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 3.2 MB/s eta 0:00:00


In [21]:
!pip install langchain_classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.6 MB/s eta 0:00:00


In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_cohere import CohereRerank
from IPython.display import Markdown, display

try:
    from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
except ImportError:
    from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever

template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Always cite the source, page_no, and headings from the context in your answer.

Context:
{context}

Question: {question}

Helpful Answer:"""

QA_PROMPT = ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Hybrid retrieves k=20 candidates; Cohere reranks and keeps top 5
compressor = CohereRerank(model="rerank-v3.5", top_n=5)
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=hybrid_retriever,
)


def format_docs(docs):
    return "\n\n".join(
        f"Source: {d.metadata.get('source')}, Page: {d.metadata.get('page_no')}, "
        f"Heading: {d.metadata.get('headings')}\nContent: {d.page_content}"
        for d in docs
    )


query = "What are the specific rules regarding VPN requirements for remote work?"

# 1. Retrieval: same hybrid_retriever pattern as the cells above
source_docs = rerank_retriever.invoke(query)

print("Retrieved chunks:")
for doc in source_docs:
    m = doc.metadata
    print(f"{m.get('source')} - page {m.get('page_no')} - {m.get('headings')}")
    print(f"   rrf_score={m.get('rrf_score'):.5f}")

# 2. Generation: pass retrieved chunks as context to the LLM
context = format_docs(source_docs)
messages = QA_PROMPT.format_messages(context=context, question=query)
response = llm.invoke(messages)

display(Markdown(f"### Question: {query}"))
display(Markdown(response.content))

Retrieved chunks:
information-security-policy.pdf - page 3 - 5.1 VPN Requirements
   rrf_score=-0.01650
remote-work-policy.pdf - page 2 - 4. Remote Work Agreement
   rrf_score=-0.01593
remote-work-policy.pdf - page 4 - 6.4 Technology Requirements
   rrf_score=-0.01541
remote-work-policy.pdf - page 1 - 3.1 General Eligibility
   rrf_score=-0.01565
remote-work-policy.pdf - page 5 - 8.3 Revocation of Remote Work Privileges
   rrf_score=-0.01572


### Question: What are the specific rules regarding VPN requirements for remote work?

Employees must use the Acme Corp VPN when working remotely on any network outside an Acme Corp office, accessing Confidential or Restricted data from any location, or connecting to any Acme Corp internal service or database. However, the VPN is not required for accessing public-facing services, such as the Acme website or public documentation (information-security-policy.pdf, Page 3, Heading: 5.1 VPN Requirements).

In [23]:
import json
import time
from pathlib import Path

# Golden dataset (same content as evaluation/eval_questions.json)
EVAL_QUESTIONS = json.loads(r"""[
  {
    "id": 1,
    "domain": "pto",
    "question": "How many PTO days do new employees receive per year?",
    "expected_answer": "New employees (0-2 years tenure) receive 15 PTO days (120 hours) per year.",
    "expected_source": "pto-and-leave-policy"
  },
  {
    "id": 2,
    "domain": "pto",
    "question": "What is the PTO carryover limit?",
    "expected_answer": "Employees may carry over a maximum of 5 unused PTO days (40 hours) into the following calendar year.",
    "expected_source": "pto-and-leave-policy"
  },
  {
    "id": 3,
    "domain": "pto",
    "question": "How much advance notice is required for a PTO request of 5 days?",
    "expected_answer": "Requests for absences of 4 or more consecutive days require at least 15 business days' advance notice.",
    "expected_source": "pto-and-leave-policy"
  },
  {
    "id": 4,
    "domain": "pto",
    "question": "How many sick days do full-time employees get?",
    "expected_answer": "Full-time employees receive 10 sick days (80 hours) per calendar year.",
    "expected_source": "pto-and-leave-policy"
  },
  {
    "id": 5,
    "domain": "pto",
    "question": "How long is the paid parental leave at Acme Corp?",
    "expected_answer": "Acme Corp provides 12 weeks of paid parental leave at 100% of regular base salary.",
    "expected_source": "pto-and-leave-policy"
  },
  {
    "id": 6,
    "domain": "security",
    "question": "What is the minimum password length required at Acme Corp?",
    "expected_answer": "All Acme Corp accounts must use passwords with a minimum length of 14 characters.",
    "expected_source": "information-security-policy"
  },
  {
    "id": 7,
    "domain": "security",
    "question": "Is SMS-based multi-factor authentication allowed?",
    "expected_answer": "No, SMS-based MFA is not permitted due to SIM-swap vulnerabilities. Approved methods are hardware security keys and authenticator apps.",
    "expected_source": "information-security-policy"
  },
  {
    "id": 8,
    "domain": "security",
    "question": "How quickly must a lost or stolen device be reported?",
    "expected_answer": "Lost or stolen devices must be reported immediately to IT Security. IT will initiate a remote wipe within 1 hour of notification.",
    "expected_source": "information-security-policy"
  },
  {
    "id": 9,
    "domain": "security",
    "question": "What encryption standard is required for restricted data?",
    "expected_answer": "Restricted data must be encrypted at rest using AES-256 encryption and in transit using TLS 1.2 or higher.",
    "expected_source": "information-security-policy"
  },
  {
    "id": 10,
    "domain": "expenses",
    "question": "What is the per-meal reimbursement limit for dinner during domestic travel?",
    "expected_answer": "The dinner reimbursement limit during domestic business travel is $50.",
    "expected_source": "expense-reimbursement-policy"
  },
  {
    "id": 11,
    "domain": "expenses",
    "question": "What is the maximum hotel rate for domestic travel?",
    "expected_answer": "Hotel stays should not exceed $200 per night for domestic travel, excluding taxes and fees.",
    "expected_source": "expense-reimbursement-policy"
  },
  {
    "id": 12,
    "domain": "expenses",
    "question": "How long do I have to submit an expense report?",
    "expected_answer": "Reimbursement requests must be submitted within 30 calendar days of incurring the expense. Expenses older than 60 days require VP-level approval.",
    "expected_source": "expense-reimbursement-policy"
  },
  {
    "id": 13,
    "domain": "expenses",
    "question": "What is the annual professional development budget per employee?",
    "expected_answer": "Each employee receives a $2,000 annual professional development budget.",
    "expected_source": "expense-reimbursement-policy"
  },
  {
    "id": 14,
    "domain": "remote_work",
    "question": "What are the core hours for remote employees?",
    "expected_answer": "All employees must be available during core hours: 10:00 AM to 3:00 PM in their designated time zone.",
    "expected_source": "remote-work-policy"
  },
  {
    "id": 15,
    "domain": "remote_work",
    "question": "What equipment does Acme Corp provide to remote workers?",
    "expected_answer": "Acme Corp provides a laptop, external 27-inch monitor, keyboard, mouse, and headset with microphone.",
    "expected_source": "remote-work-policy"
  },
  {
    "id": 16,
    "domain": "remote_work",
    "question": "How much is the monthly internet stipend for remote employees?",
    "expected_answer": "All remote and hybrid employees receive a monthly $75 internet and utilities stipend.",
    "expected_source": "remote-work-policy"
  },
  {
    "id": 17,
    "domain": "remote_work",
    "question": "Can I work from a co-working space and get reimbursed?",
    "expected_answer": "Acme Corp does not reimburse co-working space membership fees. Exceptions may be made for employees where reliable home internet is unavailable, with manager and HR approval.",
    "expected_source": "remote-work-policy"
  },
  {
    "id": 18,
    "domain": "holidays",
    "question": "Is Juneteenth a company holiday at Acme Corp?",
    "expected_answer": "Yes, Juneteenth (June 19) is one of the 12 official paid company holidays.",
    "expected_source": "holiday-schedule"
  },
  {
    "id": 19,
    "domain": "holidays",
    "question": "How many floating holidays do employees get?",
    "expected_answer": "Full-time employees receive 2 floating holidays per calendar year. Part-time employees receive 1.",
    "expected_source": "holiday-schedule"
  },
  {
    "id": 20,
    "domain": "holidays",
    "question": "What happens during the year-end office closure?",
    "expected_answer": "Acme Corp closes offices from December 26 through December 31. This is paid time off that does not count against PTO balances.",
    "expected_source": "holiday-schedule"
  },
  {
    "id": 21,
    "domain": "holidays",
    "question": "What is the holiday pay rate for essential personnel who work on holidays?",
    "expected_answer": "Essential personnel receive 1.5x their regular hourly rate plus one compensatory day off to be used within 90 days.",
    "expected_source": "holiday-schedule"
  },
  {
    "id": 22,
    "domain": "cross_domain",
    "question": "What is the 401(k) company match at Acme Corp?",
    "expected_answer": "Acme Corp matches 100% of the first 4% of the employee's salary contributed to the 401(k) plan.",
    "expected_source": "benefits-overview"
  },
  {
    "id": 23,
    "domain": "cross_domain",
    "question": "How long is the onboarding period before I can work remotely?",
    "expected_answer": "New employees must complete their 90-day onboarding period on-site before becoming eligible for hybrid or remote work arrangements.",
    "expected_source": "remote-work-policy"
  },
  {
    "id": 24,
    "domain": "cross_domain",
    "question": "What is the maximum gift value employees can accept from vendors?",
    "expected_answer": "Employees may accept nominal gifts valued at $50 or less. Gifts exceeding $50 must be reported and may need to be returned.",
    "expected_source": "code-of-conduct"
  },
  {
    "id": 25,
    "domain": "cross_domain",
    "question": "What AI tools are approved for use at Acme Corp?",
    "expected_answer": "Currently approved AI tools are GitHub Copilot (engineering, enterprise license), Google Gemini (Google Workspace enterprise tier), and custom internal AI assistants via the Acme AI Platform.",
    "expected_source": "acceptable-use-policy"
  }
]""")


def _source_stem(source) -> str:
    """Normalize metadata source to the stem used in expected_source."""
    if not source:
        return ""
    name = Path(str(source)).name
    return Path(name).stem


def _docs_sources(docs):
    return [_source_stem(d.metadata.get("source")) for d in docs]


def _first_rank(docs, expected_source: str):
    for i, d in enumerate(docs, 1):
        if _source_stem(d.metadata.get("source")) == expected_source:
            return i
    return None


def _hit(docs, expected_source: str) -> bool:
    return _first_rank(docs, expected_source) is not None


def _mrr(docs, expected_source: str) -> float:
    rank = _first_rank(docs, expected_source)
    return (1.0 / rank) if rank else 0.0

def _rerank_with_retry(question: str, max_retries: int = 6, base_sleep_s: float = 7.0):
    """Call Cohere rerank with pacing-friendly retries on trial-key 429s."""
    last_err = None
    for attempt in range(max_retries):
        try:
            return rerank_retriever.invoke(question)
        except Exception as e:
            last_err = e
            name = type(e).__name__
            msg = str(e).lower()
            is_429 = (
                name == "TooManyRequestsError"
                or "429" in str(e)
                or "too many requests" in msg
                or "rate" in msg and "limit" in msg
            )
            if not is_429 or attempt == max_retries - 1:
                raise
            wait = base_sleep_s * (attempt + 1)
            print(f"  Cohere rate limit (429). Sleeping {wait:.0f}s then retry {attempt + 2}/{max_retries}...")
            time.sleep(wait)
    raise last_err


def run_retrieval_compare(questions=None):
    """Compare hybrid RRF vs Cohere rerank on expected_source Hit@k / MRR."""
    questions = questions if questions is not None else EVAL_QUESTIONS
    if "rerank_retriever" not in globals():
        raise RuntimeError(
            "rerank_retriever is not defined. Run the Cohere rerank (or RAG generation) cell first."
        )

    rows = []
    print(f"Retrieval compare on {len(questions)} questions...")
    print("Pacing Cohere rerank at ~7s/call (trial key: 10 req/min).")
    print("-" * 72)

    for i, q in enumerate(questions, 1):
        expected = q["expected_source"]
        hybrid_docs = hybrid_retriever.invoke(q["question"])
        hybrid_top5 = hybrid_docs[:5]
        rerank_docs = _rerank_with_retry(q["question"])
        # Stay under Cohere trial limit (10 calls/min)
        time.sleep(7)

        row = {
            "id": q["id"],
            "domain": q["domain"],
            "question": q["question"],
            "expected_source": expected,
            "hybrid_hit5": _hit(hybrid_top5, expected),
            "hybrid_hit20": _hit(hybrid_docs, expected),
            "rerank_hit5": _hit(rerank_docs, expected),
            "hybrid_rank5": _first_rank(hybrid_top5, expected),
            "hybrid_rank20": _first_rank(hybrid_docs, expected),
            "rerank_rank5": _first_rank(rerank_docs, expected),
            "hybrid_mrr5": _mrr(hybrid_top5, expected),
            "rerank_mrr5": _mrr(rerank_docs, expected),
            "hybrid_sources5": _docs_sources(hybrid_top5),
            "rerank_sources5": _docs_sources(rerank_docs),
        }
        rows.append(row)

        def tag(ok):
            return "HIT " if ok else "MISS"

        print(
            f"[{i:02d}/{len(questions)}] id={q['id']} "
            f"hybrid@5={tag(row['hybrid_hit5'])} hybrid@20={tag(row['hybrid_hit20'])} "
            f"rerank@5={tag(row['rerank_hit5'])} | {q['question'][:70]}"
        )

    n = len(rows)
    hybrid_hit5_n = sum(r["hybrid_hit5"] for r in rows)
    hybrid_hit20_n = sum(r["hybrid_hit20"] for r in rows)
    rerank_hit5_n = sum(r["rerank_hit5"] for r in rows)
    hybrid_mrr = sum(r["hybrid_mrr5"] for r in rows) / n
    rerank_mrr = sum(r["rerank_mrr5"] for r in rows) / n

    summary = {
        "n": n,
        "hybrid_hit5_pct": round(hybrid_hit5_n / n * 100, 1),
        "hybrid_hit20_pct": round(hybrid_hit20_n / n * 100, 1),
        "rerank_hit5_pct": round(rerank_hit5_n / n * 100, 1),
        "hybrid_mrr5": round(hybrid_mrr, 4),
        "rerank_mrr5": round(rerank_mrr, 4),
        "hit5_delta_pp": round((rerank_hit5_n - hybrid_hit5_n) / n * 100, 1),
        "mrr5_delta": round(rerank_mrr - hybrid_mrr, 4),
        "hybrid_hit5": hybrid_hit5_n,
        "hybrid_hit20": hybrid_hit20_n,
        "rerank_hit5": rerank_hit5_n,
    }

    regressions = [r for r in rows if r["hybrid_hit5"] and not r["rerank_hit5"]]
    fixes = [r for r in rows if (not r["hybrid_hit5"]) and r["rerank_hit5"]]

    print("\n" + "=" * 72)
    print("RETRIEVAL COMPARISON SUMMARY")
    print("=" * 72)
    print(f"Hybrid Hit@5:  {summary['hybrid_hit5']}/{n} ({summary['hybrid_hit5_pct']}%)")
    print(f"Hybrid Hit@20: {summary['hybrid_hit20']}/{n} ({summary['hybrid_hit20_pct']}%)")
    print(f"Rerank Hit@5:  {summary['rerank_hit5']}/{n} ({summary['rerank_hit5_pct']}%)")
    print(f"Hit@5 delta (rerank - hybrid): {summary['hit5_delta_pp']:+.1f} pp")
    print(f"Hybrid MRR@5: {summary['hybrid_mrr5']:.4f}")
    print(f"Rerank MRR@5: {summary['rerank_mrr5']:.4f}")
    print(f"MRR@5 delta (rerank - hybrid): {summary['mrr5_delta']:+.4f}")

    print("\nPer-question:")
    for r in rows:
        print(
            f"  id={r['id']:02d} H5={'Y' if r['hybrid_hit5'] else 'N'} "
            f"H20={'Y' if r['hybrid_hit20'] else 'N'} R5={'Y' if r['rerank_hit5'] else 'N'} "
            f"ranks H5/H20/R5={r['hybrid_rank5']}/{r['hybrid_rank20']}/{r['rerank_rank5']} "
            f"| expect={r['expected_source']}"
        )
        print(f"         hybrid@5={r['hybrid_sources5']}")
        print(f"         rerank@5={r['rerank_sources5']}")

    print(f"\nRegressions (hybrid@5 HIT, rerank@5 MISS): {len(regressions)}")
    for r in regressions:
        print(f"  id={r['id']} {r['question']}")
        print(f"    expect={r['expected_source']}")
        print(f"    hybrid@5={r['hybrid_sources5']}")
        print(f"    rerank@5={r['rerank_sources5']}")

    print(f"\nFixes (hybrid@5 MISS, rerank@5 HIT): {len(fixes)}")
    for r in fixes:
        print(f"  id={r['id']} {r['question']}")
        print(f"    expect={r['expected_source']}")
        print(f"    hybrid@5={r['hybrid_sources5']}")
        print(f"    rerank@5={r['rerank_sources5']}")

    return {"summary": summary, "rows": rows, "regressions": regressions, "fixes": fixes}


retrieval_compare = run_retrieval_compare()


Retrieval compare on 25 questions...
Pacing Cohere rerank at ~7s/call (trial key: 10 req/min).
------------------------------------------------------------------------
[01/25] id=1 hybrid@5=HIT  hybrid@20=HIT  rerank@5=HIT  | How many PTO days do new employees receive per year?
[02/25] id=2 hybrid@5=HIT  hybrid@20=HIT  rerank@5=HIT  | What is the PTO carryover limit?


KeyboardInterrupt: 

In [ ]:
import json
import re
import statistics
import time
from pathlib import Path

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Golden dataset (same content as evaluation/eval_questions.json)
EVAL_QUESTIONS = json.loads(r"""[
  {
    "id": 1,
    "domain": "pto",
    "question": "How many PTO days do new employees receive per year?",
    "expected_answer": "New employees (0-2 years tenure) receive 15 PTO days (120 hours) per year.",
    "expected_source": "pto-and-leave-policy"
  },
  {
    "id": 2,
    "domain": "pto",
    "question": "What is the PTO carryover limit?",
    "expected_answer": "Employees may carry over a maximum of 5 unused PTO days (40 hours) into the following calendar year.",
    "expected_source": "pto-and-leave-policy"
  },
  {
    "id": 3,
    "domain": "pto",
    "question": "How much advance notice is required for a PTO request of 5 days?",
    "expected_answer": "Requests for absences of 4 or more consecutive days require at least 15 business days' advance notice.",
    "expected_source": "pto-and-leave-policy"
  },
  {
    "id": 4,
    "domain": "pto",
    "question": "How many sick days do full-time employees get?",
    "expected_answer": "Full-time employees receive 10 sick days (80 hours) per calendar year.",
    "expected_source": "pto-and-leave-policy"
  },
  {
    "id": 5,
    "domain": "pto",
    "question": "How long is the paid parental leave at Acme Corp?",
    "expected_answer": "Acme Corp provides 12 weeks of paid parental leave at 100% of regular base salary.",
    "expected_source": "pto-and-leave-policy"
  },
  {
    "id": 6,
    "domain": "security",
    "question": "What is the minimum password length required at Acme Corp?",
    "expected_answer": "All Acme Corp accounts must use passwords with a minimum length of 14 characters.",
    "expected_source": "information-security-policy"
  },
  {
    "id": 7,
    "domain": "security",
    "question": "Is SMS-based multi-factor authentication allowed?",
    "expected_answer": "No, SMS-based MFA is not permitted due to SIM-swap vulnerabilities. Approved methods are hardware security keys and authenticator apps.",
    "expected_source": "information-security-policy"
  },
  {
    "id": 8,
    "domain": "security",
    "question": "How quickly must a lost or stolen device be reported?",
    "expected_answer": "Lost or stolen devices must be reported immediately to IT Security. IT will initiate a remote wipe within 1 hour of notification.",
    "expected_source": "information-security-policy"
  },
  {
    "id": 9,
    "domain": "security",
    "question": "What encryption standard is required for restricted data?",
    "expected_answer": "Restricted data must be encrypted at rest using AES-256 encryption and in transit using TLS 1.2 or higher.",
    "expected_source": "information-security-policy"
  },
  {
    "id": 10,
    "domain": "expenses",
    "question": "What is the per-meal reimbursement limit for dinner during domestic travel?",
    "expected_answer": "The dinner reimbursement limit during domestic business travel is $50.",
    "expected_source": "expense-reimbursement-policy"
  },
  {
    "id": 11,
    "domain": "expenses",
    "question": "What is the maximum hotel rate for domestic travel?",
    "expected_answer": "Hotel stays should not exceed $200 per night for domestic travel, excluding taxes and fees.",
    "expected_source": "expense-reimbursement-policy"
  },
  {
    "id": 12,
    "domain": "expenses",
    "question": "How long do I have to submit an expense report?",
    "expected_answer": "Reimbursement requests must be submitted within 30 calendar days of incurring the expense. Expenses older than 60 days require VP-level approval.",
    "expected_source": "expense-reimbursement-policy"
  },
  {
    "id": 13,
    "domain": "expenses",
    "question": "What is the annual professional development budget per employee?",
    "expected_answer": "Each employee receives a $2,000 annual professional development budget.",
    "expected_source": "expense-reimbursement-policy"
  },
  {
    "id": 14,
    "domain": "remote_work",
    "question": "What are the core hours for remote employees?",
    "expected_answer": "All employees must be available during core hours: 10:00 AM to 3:00 PM in their designated time zone.",
    "expected_source": "remote-work-policy"
  },
  {
    "id": 15,
    "domain": "remote_work",
    "question": "What equipment does Acme Corp provide to remote workers?",
    "expected_answer": "Acme Corp provides a laptop, external 27-inch monitor, keyboard, mouse, and headset with microphone.",
    "expected_source": "remote-work-policy"
  },
  {
    "id": 16,
    "domain": "remote_work",
    "question": "How much is the monthly internet stipend for remote employees?",
    "expected_answer": "All remote and hybrid employees receive a monthly $75 internet and utilities stipend.",
    "expected_source": "remote-work-policy"
  },
  {
    "id": 17,
    "domain": "remote_work",
    "question": "Can I work from a co-working space and get reimbursed?",
    "expected_answer": "Acme Corp does not reimburse co-working space membership fees. Exceptions may be made for employees where reliable home internet is unavailable, with manager and HR approval.",
    "expected_source": "remote-work-policy"
  },
  {
    "id": 18,
    "domain": "holidays",
    "question": "Is Juneteenth a company holiday at Acme Corp?",
    "expected_answer": "Yes, Juneteenth (June 19) is one of the 12 official paid company holidays.",
    "expected_source": "holiday-schedule"
  },
  {
    "id": 19,
    "domain": "holidays",
    "question": "How many floating holidays do employees get?",
    "expected_answer": "Full-time employees receive 2 floating holidays per calendar year. Part-time employees receive 1.",
    "expected_source": "holiday-schedule"
  },
  {
    "id": 20,
    "domain": "holidays",
    "question": "What happens during the year-end office closure?",
    "expected_answer": "Acme Corp closes offices from December 26 through December 31. This is paid time off that does not count against PTO balances.",
    "expected_source": "holiday-schedule"
  },
  {
    "id": 21,
    "domain": "holidays",
    "question": "What is the holiday pay rate for essential personnel who work on holidays?",
    "expected_answer": "Essential personnel receive 1.5x their regular hourly rate plus one compensatory day off to be used within 90 days.",
    "expected_source": "holiday-schedule"
  },
  {
    "id": 22,
    "domain": "cross_domain",
    "question": "What is the 401(k) company match at Acme Corp?",
    "expected_answer": "Acme Corp matches 100% of the first 4% of the employee's salary contributed to the 401(k) plan.",
    "expected_source": "benefits-overview"
  },
  {
    "id": 23,
    "domain": "cross_domain",
    "question": "How long is the onboarding period before I can work remotely?",
    "expected_answer": "New employees must complete their 90-day onboarding period on-site before becoming eligible for hybrid or remote work arrangements.",
    "expected_source": "remote-work-policy"
  },
  {
    "id": 24,
    "domain": "cross_domain",
    "question": "What is the maximum gift value employees can accept from vendors?",
    "expected_answer": "Employees may accept nominal gifts valued at $50 or less. Gifts exceeding $50 must be reported and may need to be returned.",
    "expected_source": "code-of-conduct"
  },
  {
    "id": 25,
    "domain": "cross_domain",
    "question": "What AI tools are approved for use at Acme Corp?",
    "expected_answer": "Currently approved AI tools are GitHub Copilot (engineering, enterprise license), Google Gemini (Google Workspace enterprise tier), and custom internal AI assistants via the Acme AI Platform.",
    "expected_source": "acceptable-use-policy"
  }
]""")

EVAL_PROMPT = ChatPromptTemplate.from_template(
    """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Always cite sources inline using the document stem in square brackets, e.g. [pto-and-leave-policy].
Do not invent citations.

Context:
{context}

Question: {question}

Helpful Answer:"""
)

eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=10)


def _source_stem(source) -> str:
    """Normalize metadata source to the stem used in expected_source."""
    if not source:
        return ""
    name = Path(str(source)).name
    return Path(name).stem


def format_docs_for_eval(docs):
    return "\n\n".join(
        f"Source: {_source_stem(d.metadata.get('source'))}, "
        f"Page: {d.metadata.get('page_no')}, "
        f"Heading: {d.metadata.get('headings')}\n"
        f"Content: {d.page_content}"
        for d in docs
    )


def ask(question: str) -> dict:
    """Notebook RAG: hybrid + Cohere rerank + OpenAI generation."""
    docs = rerank_retriever.invoke(question)
    context = format_docs_for_eval(docs)
    messages = EVAL_PROMPT.format_messages(context=context, question=question)
    response = eval_llm.invoke(messages)
    chunks = [
        {
            "text": d.page_content,
            "source": _source_stem(d.metadata.get("source")),
        }
        for d in docs
    ]
    return {"answer": response.content, "chunks": chunks}


def check_citation_accuracy(answer: str, expected_source: str) -> bool:
    """Check if the expected source document is cited in the answer.

    Handles multiple citation formats produced by the LLM:
      - [pto-and-leave-policy]
      - [Source 2: pto-and-leave-policy]
      - Non-breaking hyphens (U+2011) used by some models
    """
    normalised = answer.replace("\u2011", "-").replace("\u2013", "-").replace("\u2014", "-")
    bracketed = re.findall(r"\[([^\]]+)\]", normalised)
    for item in bracketed:
        item_clean = re.sub(r"^Source\s+\d+:\s*", "", item, flags=re.IGNORECASE).strip()
        if item_clean == expected_source:
            return True
        if _source_stem(item_clean) == expected_source:
            return True
    return False


def check_groundedness_with_llm(question: str, answer: str, context_chunks: list) -> bool:
    """Use LLM-as-judge to check if answer is grounded in retrieved context."""
    context_text = "\n\n".join(c["text"] for c in context_chunks)

    judge_prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "You are an impartial judge evaluating whether an AI assistant's answer "
            "is fully supported by the provided context. You must respond with ONLY "
            "'GROUNDED' or 'NOT_GROUNDED'.\n\n"
            "Rules:\n"
            "- 'GROUNDED': Every factual claim in the answer is supported by the context.\n"
            "- 'NOT_GROUNDED': The answer contains information not present in or "
            "contradicted by the context.\n"
            "- If the answer correctly states it cannot find the information, that is GROUNDED.\n"
            "- Minor paraphrasing is acceptable as long as the meaning is preserved."
        )),
        ("human", (
            "Context:\n{context}\n\n"
            "Question: {question}\n\n"
            "Answer: {answer}\n\n"
            "Is this answer fully grounded in the context? Reply ONLY with "
            "'GROUNDED' or 'NOT_GROUNDED'."
        )),
    ])

    response = (judge_prompt | judge_llm).invoke({
        "context": context_text,
        "question": question,
        "answer": answer,
    })
    content = response.content.upper()
    return "GROUNDED" in content and "NOT_GROUNDED" not in content


def run_evaluation(questions=None):
    """Run the full evaluation pipeline over the golden dataset."""
    questions = questions if questions is not None else EVAL_QUESTIONS
    results = []
    latencies = []

    print(f"Running evaluation on {len(questions)} questions...")
    print("-" * 60)

    for i, q in enumerate(questions, 1):
        print(f"\n[{i}/{len(questions)}] {q['question']}")

        start = time.time()
        try:
            result = ask(q["question"])
            latency_ms = round((time.time() - start) * 1000)
        except Exception as e:
            print(f"  ERROR: {e}")
            results.append({
                "id": q["id"],
                "question": q["question"],
                "domain": q["domain"],
                "error": str(e),
                "grounded": False,
                "citation_accurate": False,
                "latency_ms": 0,
            })
            continue

        latencies.append(latency_ms)
        answer = result["answer"]
        chunks = result.get("chunks", [])

        citation_ok = check_citation_accuracy(answer, q["expected_source"])

        try:
            time.sleep(1)
            grounded = check_groundedness_with_llm(q["question"], answer, chunks)
        except Exception as e:
            print(f"  Groundedness check failed: {e}")
            grounded = None

        status = "PASS" if (grounded and citation_ok) else "PARTIAL" if (grounded or citation_ok) else "FAIL"
        print(f"  {status} | Grounded: {grounded} | Citation: {citation_ok} | Latency: {latency_ms}ms")
        print(f"  Answer preview: {answer[:120]}...")

        results.append({
            "id": q["id"],
            "question": q["question"],
            "domain": q["domain"],
            "expected_source": q["expected_source"],
            "answer": answer,
            "sources_returned": [c["source"] for c in chunks] if chunks else [],
            "grounded": grounded,
            "citation_accurate": citation_ok,
            "latency_ms": latency_ms,
        })

    valid_results = [r for r in results if "error" not in r]
    grounded_count = sum(1 for r in valid_results if r["grounded"] is True)
    grounded_total = sum(1 for r in valid_results if r["grounded"] is not None)
    citation_count = sum(1 for r in valid_results if r["citation_accurate"])

    sorted_lat = sorted(latencies)
    p95_idx = min(len(sorted_lat) - 1, int(len(sorted_lat) * 0.95)) if sorted_lat else 0

    metrics = {
        "total_questions": len(questions),
        "successful_queries": len(valid_results),
        "groundedness_pct": round(grounded_count / grounded_total * 100, 1) if grounded_total else 0,
        "citation_accuracy_pct": round(citation_count / len(valid_results) * 100, 1) if valid_results else 0,
        "latency_p50_ms": round(statistics.median(latencies)) if latencies else 0,
        "latency_p95_ms": round(sorted_lat[p95_idx]) if sorted_lat else 0,
        "latency_mean_ms": round(statistics.mean(latencies)) if latencies else 0,
    }

    domains = set(r["domain"] for r in valid_results)
    domain_metrics = {}
    for domain in sorted(domains):
        domain_results = [r for r in valid_results if r["domain"] == domain]
        d_grounded = sum(1 for r in domain_results if r["grounded"] is True)
        d_grounded_total = sum(1 for r in domain_results if r["grounded"] is not None)
        d_citation = sum(1 for r in domain_results if r["citation_accurate"])
        domain_metrics[domain] = {
            "count": len(domain_results),
            "groundedness_pct": round(d_grounded / d_grounded_total * 100, 1) if d_grounded_total else 0,
            "citation_accuracy_pct": round(d_citation / len(domain_results) * 100, 1) if domain_results else 0,
        }

    output = {
        "metrics": metrics,
        "domain_metrics": domain_metrics,
        "results": results,
    }

    print("\n" + "=" * 60)
    print("EVALUATION SUMMARY")
    print("=" * 60)
    print(f"Total questions: {metrics['total_questions']}")
    print(f"Successful queries: {metrics['successful_queries']}")
    print(f"Groundedness: {metrics['groundedness_pct']}%")
    print(f"Citation Accuracy: {metrics['citation_accuracy_pct']}%")
    print(f"Latency p50: {metrics['latency_p50_ms']}ms")
    print(f"Latency p95: {metrics['latency_p95_ms']}ms")
    print(f"Latency mean: {metrics['latency_mean_ms']}ms")
    print("\nPer-domain breakdown:")
    for domain, dm in domain_metrics.items():
        print(
            f"  {domain}: {dm['count']} questions | "
            f"Groundedness: {dm['groundedness_pct']}% | "
            f"Citation Accuracy: {dm['citation_accuracy_pct']}%"
        )

    # Per-question pass/fail table
    print("\nPer-question:")
    for r in results:
        if "error" in r:
            print(f"  [ERR ] id={r['id']} {r['question']}")
            continue
        ok = r["grounded"] is True and r["citation_accurate"]
        tag = "PASS" if ok else "FAIL"
        print(
            f"  [{tag}] id={r['id']} grounded={r['grounded']} "
            f"citation={r['citation_accurate']} | {r['question']}"
        )

    return output


eval_output = run_evaluation()


Running evaluation on 25 questions...
------------------------------------------------------------

[1/25] How many PTO days do new employees receive per year?
  PASS | Grounded: True | Citation: True | Latency: 2082ms
  Answer preview: New employees at Acme Corp receive 15 days (120 hours) of PTO per year during their first two years of employment [pto-a...

[2/25] What is the PTO carryover limit?
  PASS | Grounded: True | Citation: True | Latency: 3385ms
  Answer preview: The PTO carryover limit is a maximum of 5 unused PTO days (40 hours) into the following calendar year. Any PTO balance e...

[3/25] How much advance notice is required for a PTO request of 5 days?
  PASS | Grounded: True | Citation: True | Latency: 2259ms
  Answer preview: A PTO request for 5 days requires at least 15 business days' advance notice [pto-and-leave-policy]....

[4/25] How many sick days do full-time employees get?
  PASS | Grounded: True | Citation: True | Latency: 2054ms
  Answer preview: Full-time em

In [ ]:
!pip install -q -U "ragas" "openai" "langchain-core" "langchain-openai" "langchain-community"
!pip install -q --force-reinstall "instructor>=1.7.0,<1.8"

import asyncio
import sys
import types
from pathlib import Path

# Drop stale instructor imports from a previous failed run
for _name in list(sys.modules):
    if _name == "instructor" or _name.startswith("instructor."):
        del sys.modules[_name]

from openai import AsyncOpenAI

# Some ragas versions import removed langchain_community VertexAI paths at import time.
# Stub them so ragas can load without google-vertex packages.
def _stub_module(name: str, **attrs):
    mod = types.ModuleType(name)
    for k, v in attrs.items():
        setattr(mod, k, v)
    sys.modules[name] = mod
    return mod

try:
    from langchain_community.chat_models.vertexai import ChatVertexAI  # noqa: F401
except ModuleNotFoundError:
    class ChatVertexAI:  # noqa: N801
        pass

    _stub_module("langchain_community.chat_models.vertexai", ChatVertexAI=ChatVertexAI)
    try:
        import langchain_community.chat_models as _cm
        _cm.vertexai = sys.modules["langchain_community.chat_models.vertexai"]
    except Exception:
        pass

try:
    from langchain_community.llms import VertexAI  # noqa: F401
except Exception:
    class VertexAI:  # noqa: N801
        pass

    # Ensure langchain_community.llms.VertexAI attribute exists if package is present
    try:
        import langchain_community.llms as _llms
        if not hasattr(_llms, "VertexAI"):
            _llms.VertexAI = VertexAI
    except Exception:
        _stub_module("langchain_community.llms", VertexAI=VertexAI)

from ragas import Dataset, experiment
from ragas.llms import llm_factory
from ragas.metrics import DiscreteMetric

# --- Drive corpus check (PDFs live on Google Drive, not a local Windows path) ---
assert "PDF_FOLDER" in globals(), "Run the config cell that sets PDF_FOLDER first."
pdf_paths = sorted(Path(PDF_FOLDER).glob("*.pdf"))
assert pdf_paths, (
    f"No PDFs under {PDF_FOLDER}. Mount Drive (drive.mount) and confirm the folder path."
)
print(f"Evaluating RAG over {len(pdf_paths)} PDFs from {PDF_FOLDER}")
for path in pdf_paths:
    print(f"  - {path.name}")

# --- Golden Q&A about those PDFs ---
assert "EVAL_QUESTIONS" in globals() and EVAL_QUESTIONS, (
    "Run the retrieval-compare cell first so EVAL_QUESTIONS is defined."
)
assert "hybrid_retriever" in globals(), "Run the hybrid retriever cell first."

# Prompt / LLM for naive generation (reuse golden-eval helpers when present)
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

if "EVAL_PROMPT" not in globals():
    EVAL_PROMPT = ChatPromptTemplate.from_template(
        """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Always cite sources inline using the document stem in square brackets, e.g. [pto-and-leave-policy].
Do not invent citations.

Context:
{context}

Question: {question}

Helpful Answer:"""
    )

if "eval_llm" not in globals():
    eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def _format_docs_for_ragas(docs):
    if "format_docs_for_eval" in globals():
        return format_docs_for_eval(docs)
    if "_source_stem" in globals():
        stem = _source_stem
    else:
        def stem(source):
            if not source:
                return ""
            name = Path(str(source)).name
            return Path(name).stem
    return "\n\n".join(
        f"Source: {stem(d.metadata.get('source'))}, "
        f"Page: {d.metadata.get('page_no')}, "
        f"Heading: {d.metadata.get('headings')}\n"
        f"Content: {d.page_content}"
        for d in docs
    )


def naive_rag_query(question: str, k: int = 5) -> dict:
    """Naive RAG: retrieve once (hybrid top-k), then generate. No agent loop."""
    docs = hybrid_retriever.invoke(question)[:k]
    context = _format_docs_for_ragas(docs)
    messages = EVAL_PROMPT.format_messages(context=context, question=question)
    answer = eval_llm.invoke(messages).content
    return {
        "answer": answer,
        "contexts": [d.page_content for d in docs],
        "sources": [
            (_source_stem(d.metadata.get("source")) if "_source_stem" in globals() else d.metadata.get("source"))
            for d in docs
        ],
    }


# --- Ragas correctness metric (pass/fail), from the improve-RAG guide ---
correctness_metric = DiscreteMetric(
    name="correctness",
    prompt="""Compare the model response to the expected answer and determine if it's correct.

Consider the response correct if it:
1. Contains the key information from the expected answer
2. Is factually accurate based on the provided context
3. Adequately addresses the question asked

Return 'pass' if the response is correct, 'fail' if it's incorrect.

Question: {question}
Expected Answer: {expected_answer}
Model Response: {response}

Evaluation:""",
    allowed_values=["pass", "fail"],
)

# Prefer ragas llm_factory; fall back if instructor/ragas versions disagree
try:
    ragas_llm = llm_factory("gpt-4o-mini", client=AsyncOpenAI())
except Exception as e:
    print(f"llm_factory failed ({e}); falling back to LangchainLLMWrapper.")
    from ragas.llms import LangchainLLMWrapper
    from langchain_openai import ChatOpenAI as _ChatOpenAI
    ragas_llm = LangchainLLMWrapper(_ChatOpenAI(model="gpt-4o-mini", temperature=0))


# Build Ragas Dataset from golden Q&A
ragas_root = Path("/content/ragas_eval")
ragas_root.mkdir(parents=True, exist_ok=True)
ragas_dataset = Dataset(name="acme_policy_qa", backend="local/csv", root_dir=str(ragas_root))
for q in EVAL_QUESTIONS:
    ragas_dataset.append(
        {
            "id": q["id"],
            "domain": q.get("domain", ""),
            "question": q["question"],
            "expected_answer": q["expected_answer"],
            "expected_source": q.get("expected_source", ""),
        }
    )
ragas_dataset.save()
print(f"Ragas dataset rows: {len(EVAL_QUESTIONS)}")


@experiment()
async def evaluate_rag(row, llm):
    """Run naive RAG + Ragas correctness on one golden question."""
    question = row["question"]
    rag_response = naive_rag_query(question, k=5)
    model_response = rag_response.get("answer", "")

    score = await correctness_metric.ascore(
        question=question,
        expected_answer=row["expected_answer"],
        response=model_response,
        llm=llm,
    )

    contexts = rag_response.get("contexts", [])
    truncated = [
        (c[:200] + "...") if len(c) > 200 else c for c in contexts
    ]

    return {
        **row,
        "model_response": model_response,
        "correctness_score": score.value,
        "correctness_reason": getattr(score, "reason", None),
        "retrieved_sources": rag_response.get("sources", []),
        "retrieved_contexts": truncated,
    }


async def run_ragas_evaluation():
    exp_name = "naive_hybrid_ragas"
    results = await evaluate_rag.arun(
        ragas_dataset,
        name=exp_name,
        llm=ragas_llm,
    )
    return results


async def run_ragas_evaluation_sync_fallback():
    """Plain async loop if Dataset/experiment.arun is unavailable."""
    rows = []
    for i, q in enumerate(EVAL_QUESTIONS, 1):
        question = q["question"]
        print(f"[{i}/{len(EVAL_QUESTIONS)}] {question}")
        rag_response = naive_rag_query(question, k=5)
        model_response = rag_response.get("answer", "")
        score = await correctness_metric.ascore(
            question=question,
            expected_answer=q["expected_answer"],
            response=model_response,
            llm=ragas_llm,
        )
        contexts = rag_response.get("contexts", [])
        truncated = [(c[:200] + "...") if len(c) > 200 else c for c in contexts]
        rows.append({
            "id": q["id"],
            "domain": q.get("domain", ""),
            "question": question,
            "expected_answer": q["expected_answer"],
            "expected_source": q.get("expected_source", ""),
            "model_response": model_response,
            "correctness_score": score.value,
            "correctness_reason": getattr(score, "reason", None),
            "retrieved_sources": rag_response.get("sources", []),
            "retrieved_contexts": truncated,
        })
    return rows


try:
    ragas_results = await run_ragas_evaluation()
except Exception as e:
    print(f"experiment.arun failed ({type(e).__name__}: {e}); using fallback loop.")
    ragas_results = await run_ragas_evaluation_sync_fallback()

# Summary
rows = list(ragas_results) if ragas_results is not None else []
pass_count = sum(1 for r in rows if str(r.get("correctness_score", "")).lower() == "pass")
total = len(rows)
pass_rate = (pass_count / total * 100) if total else 0.0

print("\n" + "=" * 60)
print("RAGAS NAIVE RAG SUMMARY")
print("=" * 60)
print(f"Corpus: {PDF_FOLDER} ({len(pdf_paths)} PDFs)")
print(f"Correctness: {pass_count}/{total} passed ({pass_rate:.1f}%)")

fails = [r for r in rows if str(r.get("correctness_score", "")).lower() != "pass"]
if fails:
    print("\nFailures:")
    for r in fails:
        print(f"  id={r.get('id')} | {r.get('question')}")
        print(f"    score={r.get('correctness_score')} reason={r.get('correctness_reason')}")
        print(f"    answer preview: {str(r.get('model_response', ''))[:160]}...")
else:
    print("\nNo failures.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 35.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.8/344.8 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_2784/1363182653.py:158: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(_ChatOpenAI(model="gpt-4o-mini", temperature=0))


llm_factory failed (Failed to initialize openai client with instructor adapter. Ensure you've created a valid openai client.
Error: Failed to patch openai client with Instructor: No module named 'instructor.v2'); falling back to LangchainLLMWrapper.
Ragas dataset rows: 25


Running experiment:   0%|          | 0/25 [00:00<?, ?it/s]ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-7' coro=<AsyncClient.aclose() done, defined at /usr/local/lib/python3.13/dist-packages/httpx/_client.py:1978> exception=ImportError("cannot import name 'get_coro_name' from 'anyio.abc._tasks' (/usr/local/lib/python3.13/dist-packages/anyio/abc/_tasks.py)")>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/anyio/_core/_eventloop.py", line 204, in get_async_backend
    return loaded_backends[asynclib_name]
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
KeyError: 'asyncio'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/httpx/_client.py", line 1985, in aclose
    await self._transport.aclose()
  File "/usr/local/lib/python3.13/dist-packages/httpx/_transports/default.py", line 406, in aclose
    await self._pool.aclose()


RAGAS NAIVE RAG SUMMARY
Corpus: /content/drive/MyDrive/research_papers (11 PDFs)
Correctness: 0/0 passed (0.0%)

No failures.
